**Table of contents**<a id='toc0_'></a>    
- [Title:](#toc1_1_1_)    
    - [Authors:](#toc1_1_2_)    
    - [Abstract:](#toc1_1_3_)    
    - [References](#toc1_1_4_)    
    - [Assumption](#toc1_1_5_)    
    - [Mathematical basis](#toc1_1_6_)    
    - [Numerical method](#toc1_1_7_)    
        - [Step 1) Simulate $X_u|X_t$ using the noncentral $\chi^2$ distribution](#toc1_1_7_1_1_)    
        - [Step 2) $\int_t^u \frac{ds}{X_s}$ Given $X_u, X_t$](#toc1_1_7_1_2_)    
        - [Step 3) Exact method: Simulate $V_T$ with CDF by Laplace transform](#toc1_1_7_1_3_)    
        - [Step 3) Almost exact method: Simulate $V_T$ with assumed distribution](#toc1_1_7_1_4_)    
        - [Finally get the option price](#toc1_1_7_1_5_)    
  - [Case I: Eq. (4.2) in Baldeaux (2012)](#toc1_2_)    
  - [Case II: Set 2 in Kouarfate et al. (2021)](#toc1_3_)    
  - [Case III:in Kouarfate et al. (2021)](#toc1_4_)    
  - [Case IV:](#toc1_5_)    
  - [Pricing with Time Discreteization using Euler/Milstein scheme, Exact Stepping, Almost Exact Stepping](#toc1_6_)    
  - [Pricing with Exact Simulation](#toc1_7_)    
  - [Pricing with IG approximation (Almost Exact Simulation)](#toc1_8_)    
- [Pricing with FFT method](#toc2_)    
- [Pricing with approximation IV](#toc3_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

### <a id='toc1_1_1_'></a>[Title:](#toc0_)
__Exact simulation and Almost exact simulation of the 3/2 model__

### <a id='toc1_1_2_'></a>[Authors:](#toc0_)
* Jan Baldeaux
* Choi J, Kwok YK
* Medvedev, A., Scaillet, O.

### <a id='toc1_1_3_'></a>[Abstract:](#toc0_)
* Baldeaux J (2012) Exact simulation of the 3/2 model. Int J Theor Appl Finan 15:1250032. https://doi.org/10.1142/S021902491250032X

This paper discusses the exact simulation of the stock price process underlying the 3/2 model. Using a result derived by Craddock and Lennox using Lie Symmetry Analysis, we adapt the Broadie-Kaya algorithm for the simulation of affine processes to the 3/2 model. We also discuss variance reduction techniques and find that conditional Monte Carlo techniques combined with quasi-Monte Carlo point sets result in significant variance reductions.

* Choi J, Kwok YK (2024) Simulation schemes for the Heston model with Poisson conditioning. European Journal of Operational Research 314(1):363–376. https://doi.org/10.1016/j.ejor.2023.10.048

Exact simulation schemes under the Heston stochastic volatility model (e.g., Broadie–Kaya and Glasserman–Kim) suffer from computationally expensive modified Bessel function evaluations. We propose a new exact simulation scheme without the modified Bessel function, based on the observation that the conditional integrated variance can be simplified when conditioned by the Poisson variate used for simulating the terminal variance. Our approach also enhances the low-bias and time discretization schemes, which are suitable for pricing derivatives with frequent monitoring. Extensive numerical tests reveal the good performance of the new simulation schemes in terms of accuracy, efficiency, and reliability when compared with existing methods.

* Medvedev, A., & Scaillet, O. (2007). Approximation and calibration of short-term implied volatilities under jump-diffusion stochastic volatility. The Review of Financial Studies, 20(2), 427-459.https://doi.org/10.1093/rfs/hhl013

We derive an asymptotic expansion formula for option implied volatility under a two factor jump-diffusion stochastic volatility model when time-to-maturity is small. We further propose a simple calibration procedure of an arbitrary parametric model to short-term near-the-money implied volatilities. An important advantage of our approximation is that it is free of the unobserved spot volatility. Therefore, the model can be calibrated on option data pooled across different calendar dates to extract information from the dynamics of the implied volatility smile. An example of calibration to a sample of S&P 500 option prices is provided. 


### <a id='toc1_1_4_'></a>[References](#toc0_)
* Baldeaux J (2012) Exact simulation of the 3/2 model. Int J Theor Appl Finan 15:1250032.
* Kouarfate IR, Kouritzin MA, MacKay A (2021) Explicit Solution Simulation Method for the 3/2 Model. In: Hernández‐Hernández D, Leonardi F, Mena RH, Pardo Millán JC (eds) Advances in Probability and Mathematical Statistics. Springer International Publishing, Cham, pp 123–145
* M. Jeanblanc, M. Yor and M. Chesney, Mathematical Methods for Financial Markets (Springer Finance, Springer, 2009).
* Choi J, Kwok YK (2024) Simulation schemes for the Heston model with Poisson conditioning. European Journal of Operational Research 314(1):363–376.
* Medvedev, A., & Scaillet, O. (2007). Approximation and calibration of short-term implied volatilities under jump-diffusion stochastic volatility. The Review of Financial Studies, 20(2), 427-459.

### <a id='toc1_1_5_'></a>[Assumption](#toc0_)

According to Baldeaux(2013), The dynamics of the stock price under the 3/2 model under the risk-neutral measure are given by

$$
 \frac{dS_t}{S_t} = rdt + \sqrt{V_t}\rho dW_t^1 + \sqrt{V_t}\sqrt{1-\rho^2}dW_t^2 \tag{1}
$$

$$
 \frac{dV_t}{V_t} = \kappa (\theta - V_t)dt + \epsilon \sqrt{V_t}dW_t^1
$$

which is equivalent to

$$
 dV_t = \kappa V_t (\theta - V_t)dt + \epsilon V_t^{3/2}dW_t^1 \tag{2}
$$

where $W_t^1$ and $W_t^2$ are independent Brownian motions. Regarding the parameters, $r$ represents the constant interest rate, $\rho$ the instantaneous correlation between the return on the stock and the variance process and $\epsilon$ governs the volatility of volatility.The speed of mean reversion is given by $\kappa V_t$ and $\theta$ denotes the long-run mean of the variance process.

### <a id='toc1_1_6_'></a>[Mathematical basis](#toc0_)

Defining $X_t = \frac{1}{V_t}$, we obtain

$$
dX_t = (\kappa + \epsilon^2 - \kappa\theta X_t)dt - \epsilon \sqrt{X_t}dW_t^1 \tag{3}
$$
Hence, using the process $X_t$, we obtain the following dynamics for the stock price, where $u > t$

$$
S_u = S_t \exp\lbrace r(u-t) - 1/2 \int_t^u(X_s)^{-1}ds + \rho \int_t^u({\sqrt{X_s})^{-1}dW_s^1}\rbrace \exp \lbrace \sqrt{1-\rho^2} \int_t^u(\sqrt{X_s})^{-1} dW_s^2\rbrace \tag{4}
$$
From Baldeaux(2013), study $\log(X_t)$

$$
\int_t^u({\sqrt{X_s})^{-1}dW_s^1} = \frac{1}{\epsilon} (log(\frac{X_t}{X_u}) + (k + \frac{\epsilon^2}{2})\int_{t}^{u}\frac{ds}{X_s}-k\theta(u-t)) \tag{5}
$$

Therefore, the only thing we need to know is the distribution of $X_t$ and $\int_{t}^{u}\frac{ds}{X_s}$ conditional on $X_t$

### <a id='toc1_1_7_'></a>[Numerical method](#toc0_)
Using Broadie-Kaya algorithm, we specify the simulation as 3 steps

##### <a id='toc1_1_7_1_1_'></a>[Step 1) Simulate $X_u|X_t$ using the noncentral $\chi^2$ distribution](#toc0_)
$X_u$ is distributed as a noncentral $ \chi^2 $ distribution

$$
\frac{X_u{\rm exp}\lbrace \kappa \theta (u-t) \rbrace}{c(u-t)} \sim \chi^2(\delta, \alpha) \tag{6}
$$

where
$$
\delta = \frac{4(\kappa + \epsilon^2)}{\epsilon^2}, \quad \alpha = \frac{X_t}{c(u-t)}, \quad c(t) = \frac{\epsilon^2({\rm exp}\lbrace \kappa\theta u \rbrace - 1)}{4\kappa\theta}
$$

##### <a id='toc1_1_7_1_2_'></a>[Step 2) $\int_t^u \frac{ds}{X_s}$ Given $X_u, X_t$](#toc0_)
We first derive the characteristic function of $\int_u^t \frac{ds}{X_s}$, which is provided in Baldeaux(2013)

$$
E\left({\rm exp}\left\lbrace -a^* \int_0^t \frac{ds}{X_s} \ \bigg| \ X_t \right\rbrace \right) = \frac{I_{\sqrt{\nu^2+8a/\epsilon^2}}\left(-\frac{j\sqrt{X_tX_u}}{{\rm sinh}\left(j\Delta\right)}\right)}{I_{\nu}\left(-\frac{j\sqrt{X_tX_u}}{{\rm sinh}\left(j\Delta\right)}\right)}\tag{7}
$$

where $j=-\frac{2\kappa\theta}{\epsilon^2}$, $\Delta=\frac{u\epsilon^2}{4}-\frac{t\epsilon^2}{4}$, $v=\frac{n}{2}-1$.

##### <a id='toc1_1_7_1_3_'></a>[Step 3) Exact method: Simulate $V_T$ with CDF by Laplace transform](#toc0_)
Use Laplace transform to get the CDF of $\int_t^u \frac{ds}{X_s}$, then get the price.

##### <a id='toc1_1_7_1_4_'></a>[Step 3) Almost exact method: Simulate $V_T$ with assumed distribution](#toc0_)
Then we use the characteristic function to generate moment $M1$, $M2$. For simplicity, we assume that $\int_t^u \frac{ds}{X_s}$ follows the Inverse-Gaussian distribution / Gamma distribution / Log-normal distribution, then we can simulate $\int_t^u \frac{ds}{X_s}$.

##### <a id='toc1_1_7_1_5_'></a>[Finally get the option price](#toc0_)
$$
log(S_u) \sim N(log(S_t)+r(u-t)-\frac{1}{2}\int_t^u \frac{ds}{X_s}+\rho\int_t^u({\sqrt{X_s})^{-1}dW_s^1},  \sigma^2(t,u))\tag{8}
$$
where 
$$
\sigma^2(t, u) = (1-\rho^2)\int_t^u \frac{ds}{X_s}
$$
The option price is $C_{BS}(K,S_u, \sigma(t, u))$

In [3]:
%load_ext autoreload
%autoreload 2

In [16]:
import numpy as np
import pandas as pd
import time
import scipy.stats as spst
import scipy.special as spsp
import copy 
#import mp math as mp

import sys
sys.path.insert(sys.path.index("")+1, "C:/Users/27261/Desktop/3_Courses in PHBS/3_09_AppliedStochasticProcess/Project_sv32_EMC")
import pyfeng as pf
import pyfeng.ex as pfex
np.set_printoptions(precision=4)
from utils import *
seed_everything()
case_names = case_dict.keys()

全局随机数种子已锁定为: 123456


## <a id='toc1_6_'></a>[Pricing with Time Discretization using Euler/Milstein scheme, Exact Stepping, Almost Exact Stepping](#toc0_)

In [17]:
# Milstein cannot price Case III and Case VII
case_name = ["Case VI"]
run_valuation_for_single_model(pfex.Sv32McTimeStep, 1, case_name, case_dict)


========== 正在运行模型: Sv32McTimeStep ==========
Bias: [inf inf inf inf inf] | dt: 0.002
[Case VI]: atm only 
 运行耗时: 3.330458 秒
------------------------------


c:\Users\27261\Desktop\3_Courses in PHBS\3_09_AppliedStochasticProcess\Project_sv32_EMC\pyfeng\sv32_mc2.py:151: RuntimeWarning: overflow encountered in exp
  """


In [18]:
# Exact Stepping with 1 / NCX2, cannot price Case V and Case VII
run_valuation_for_single_model(pfex.Sv32McTimeStep, 2, case_names, case_dict)


========== 正在运行模型: Sv32McTimeStep ==========
Bias: 0.0007256385500868934 | dt: 0.002
[Case I]: Eq. (4.2) in Baldeaux (2012), ATM 
 运行耗时: 4.145985 秒
------------------------------
[Case II]: Set 2 in Kouarfate et al. (2021), ATM 
 运行耗时: 2.097303 秒


KeyboardInterrupt: 

In [ ]:
# Almost Exact Stepping with Poison-Gamma distribution, cannot price Case II, Case III, Case VI and Case VII
run_valuation_for_single_model(pfex.Sv32McTimeStep, 3, case_names, case_dict)


========== 正在运行模型: Sv32McTimeStep ==========
Bias: -8.394706365394411e-05 | dt: 0.0002
[Case I]: Eq. (4.2) in Baldeaux (2012), ATM 
 运行耗时: 60.486132 秒
------------------------------
Bias: [-0.0135 -0.0121 -0.0103] | dt: 0.0002
[Case II]: Set 2 in Kouarfate et al. (2021), ATM 
 运行耗时: 31.949373 秒
------------------------------
Bias: [-0.0397 -0.0374 -0.0347] | dt: 0.0002
[Case III]: in Kouarfate et al. (2021), ATM 
 运行耗时: 30.242102 秒
------------------------------
Bias: [0.0027 0.0703 0.0033 0.0033 0.0032] | dt: 0.0002
[Case IV]: Near expiration and atm 
 运行耗时: 5.983264 秒
------------------------------
Bias: [0.0072 0.0072 0.0054] | dt: 0.0002
[Case V]: Near expiration only 
 运行耗时: 5.963417 秒
------------------------------
Bias: [0.0183 0.018  0.0178 0.0175 0.0173] | dt: 0.0002
[Case VI]: atm only 
 运行耗时: 61.284346 秒
------------------------------
Bias: [-0.0392 -0.0362 -0.0338] | dt: 0.0002
[Case VII]: Lewis AL (2000) Option valuation under stochastic volatility: with Mathematica code.

In [ ]:
# QE method, cannot price Case VI
run_valuation_for_single_model(pfex.Sv32McTimeStep, 4, case_names, case_dict)


========== 正在运行模型: Sv32McTimeStep ==========
Bias: -0.00014108583206956515 | dt: 0.0002
[Case I]: Eq. (4.2) in Baldeaux (2012), ATM 
 运行耗时: 68.278836 秒
------------------------------
Bias: [-0.0136 -0.0164 -0.0168] | dt: 0.0002
[Case II]: Set 2 in Kouarfate et al. (2021), ATM 
 运行耗时: 33.579577 秒
------------------------------
Bias: [-0.0202 -0.0228 -0.0207] | dt: 0.0002
[Case III]: in Kouarfate et al. (2021), ATM 
 运行耗时: 33.558054 秒
------------------------------
Bias: [-7.7072e-04  6.6927e-02  1.3558e-05  8.2636e-05  6.5103e-05] | dt: 0.0002
[Case IV]: Near expiration and atm 
 运行耗时: 6.732067 秒
------------------------------
Bias: [0.0006 0.0007 0.0009] | dt: 0.0002
[Case V]: Near expiration only 
 运行耗时: 6.760751 秒
------------------------------
Bias: [3.0574e+21 3.0574e+21 3.0574e+21 3.0574e+21 3.0574e+21] | dt: 0.0002
[Case VI]: atm only 
 运行耗时: 66.869347 秒
------------------------------
Bias: [-0.0197 -0.0216 -0.0198] | dt: 0.0002
[Case VII]: Lewis AL (2000) Option valuation under

## <a id='toc1_7_'></a>[Pricing with Exact Simulation](#toc0_)
## Exact simulation **cannot** price case II to case VII

In [ ]:
# Exact MC cannot price Case II to Case VII
run_valuation_for_single_model(pfex.Sv32McBaldeaux2012Exact, None, case_names, case_dict)


========== 正在运行模型: Sv32McBaldeaux2012Exact ==========
Bias: -1.5712420661784243e-05 | dt: None
[Case I]: Eq. (4.2) in Baldeaux (2012), ATM 
 运行耗时: 69.085747 秒
------------------------------


C:\Users\27261\AppData\Roaming\Python\Python311\site-packages\scipy\optimize\_zeros_py.py:482: RuntimeWarning: some failed to converge after 50 iterations
  warnings.warn(msg, RuntimeWarning)
c:\Users\27261\Desktop\3_Courses in PHBS\3_09_AppliedStochasticProcess\Project_sv32_EMC\pyfeng\sv32_mc2.py:151: RuntimeWarning: overflow encountered in exp
  np.exp(self.rho * spot_cond, out=spot_cond)
c:\Users\27261\Desktop\3_Courses in PHBS\3_09_AppliedStochasticProcess\Project_sv32_EMC\pyfeng\sv32_mc2.py:152: RuntimeWarning: invalid value encountered in sqrt
  sigma_cond = np.sqrt(
c:\Users\27261\Desktop\3_Courses in PHBS\3_09_AppliedStochasticProcess\Project_sv32_EMC\pyfeng\bsm.py:55: RuntimeWarning: divide by zero encountered in log
  d1 = np.log(fwd/strike)/sigma_std


Bias: [nan nan nan] | dt: None
[Case II]: Set 2 in Kouarfate et al. (2021), ATM 
 运行耗时: 101.066009 秒
------------------------------
Bias: [nan nan nan] | dt: None
[Case III]: in Kouarfate et al. (2021), ATM 
 运行耗时: 101.464271 秒
------------------------------


c:\Users\27261\Desktop\3_Courses in PHBS\3_09_AppliedStochasticProcess\Project_sv32_EMC\pyfeng\sv32_mc2.py:369: RuntimeWarning: divide by zero encountered in log
  ln_sig = np.sqrt(np.log(1 + var / m1**2))
c:\Users\27261\Desktop\3_Courses in PHBS\3_09_AppliedStochasticProcess\Project_sv32_EMC\pyfeng\sv32_mc2.py:369: RuntimeWarning: invalid value encountered in sqrt
  ln_sig = np.sqrt(np.log(1 + var / m1**2))


Bias: [nan nan nan nan nan] | dt: None
[Case IV]: Near expiration and atm 
 运行耗时: 108.214504 秒
------------------------------
Bias: [nan nan nan] | dt: None
[Case V]: Near expiration only 
 运行耗时: 107.079444 秒
------------------------------
Bias: [nan nan nan nan nan] | dt: None
[Case VI]: atm only 
 运行耗时: 100.245648 秒
------------------------------
Bias: [nan nan nan] | dt: None
[Case VII]: Lewis AL (2000) Option valuation under stochastic volatility: with Mathematica code. Finance Press 
 运行耗时: 99.612390 秒
------------------------------


## <a id='toc1_8_'></a>[Pricing with IG approximation (Almost Exact Simulation)](#toc0_)
## Almost exact simulation **cannot** price case II to V, and case VII

In [ ]:
model = pfex.Sv32McChoiKwok2023Ig
model.dist = "ig"
run_valuation_for_single_model(pfex.Sv32McChoiKwok2023Ig, None, case_names, case_dict)


========== 正在运行模型: Sv32McChoiKwok2023Ig ==========
Bias: 0.0013260257417284649 | dt: None
[Case I]: Eq. (4.2) in Baldeaux (2012), ATM 
 运行耗时: 1.930526 秒
------------------------------
Bias: [-8.2148 -6.2132 -4.3606] | dt: None
[Case II]: Set 2 in Kouarfate et al. (2021), ATM 
 运行耗时: 0.190566 秒
------------------------------
Bias: [-11.6418  -8.9603  -6.6927] | dt: None
[Case III]: in Kouarfate et al. (2021), ATM 
 运行耗时: 0.201198 秒
------------------------------
Bias: [-5.5587 -5.2872 -5.1345 -4.903  -4.6638] | dt: None
[Case IV]: Near expiration and atm 
 运行耗时: 13.520441 秒
------------------------------
Bias: [-5.2551 -5.2586 -6.3558] | dt: None
[Case V]: Near expiration only 
 运行耗时: 13.718989 秒
------------------------------
Bias: [-0.0698 -0.0663 -0.0628 -0.0594 -0.0563] | dt: None
[Case VI]: atm only 
 运行耗时: 0.180425 秒
------------------------------
Bias: [-11.6413  -8.9591  -6.6918] | dt: None
[Case VII]: Lewis AL (2000) Option valuation under stochastic volatility: with Mathemati

In [ ]:
model = pfex.Sv32McChoiKwok2023Ig
model.dist = "ga"
run_valuation_for_single_model(model, None, case_names, case_dict)


========== 正在运行模型: Sv32McChoiKwok2023Ig ==========
Bias: 0.0008747258145394565 | dt: None
[Case I]: Eq. (4.2) in Baldeaux (2012), ATM 
 运行耗时: 1.851818 秒
------------------------------
Bias: [-7.9563 -5.9942 -4.1946] | dt: None
[Case II]: Set 2 in Kouarfate et al. (2021), ATM 
 运行耗时: 0.179537 秒
------------------------------
Bias: [-11.6729  -8.9749  -6.6992] | dt: None
[Case III]: in Kouarfate et al. (2021), ATM 
 运行耗时: 0.190856 秒
------------------------------
Bias: [38.3066 37.9138 37.3681 36.8724 36.3609] | dt: None
[Case IV]: Near expiration and atm 
 运行耗时: 12.440652 秒
------------------------------
Bias: [43.8307 43.8272 41.9916] | dt: None
[Case V]: Near expiration only 
 运行耗时: 12.178911 秒
------------------------------
Bias: [-0.1655 -0.1583 -0.1515 -0.145  -0.1389] | dt: None
[Case VI]: atm only 
 运行耗时: 0.180277 秒
------------------------------
Bias: [-11.6724  -8.9737  -6.6983] | dt: None
[Case VII]: Lewis AL (2000) Option valuation under stochastic volatility: with Mathemati

# <a id='toc2_'></a>[Pricing with FFT method](#toc0_)
## FFT method **cannot** price case IV, case V, case VI

In [ ]:
case_names_cut = ["Case IV", "Case V", "Case VI"]
run_valuation_for_single_model(pf.sv_fft.Sv32Fft, None, "",case_names, case_dict, {})


========== 正在运行模型: Sv32Fft -  ==========
[Case I]: 运行耗时: 0.371528 秒 | dt: N/A
[Case II]: 运行耗时: 0.187956 秒 | dt: N/A
[Case III]: 运行耗时: 0.271939 秒 | dt: N/A
[Case IV]: 运行耗时: 13.105926 秒 | dt: N/A


KeyboardInterrupt: 

In [ ]:
run_valuation_for_single_model(pf.sv_fft.Sv32FourierCos, None, case_names, case_dict)


========== 正在运行模型: Sv32FourierCos ==========
Bias: 6.55534504216404e-05 | dt: None
[Case I]: Eq. (4.2) in Baldeaux (2012), ATM 
 运行耗时: 0.354158 秒
------------------------------
Bias: [-0.0012 -0.0007 -0.0017] | dt: None
[Case II]: Set 2 in Kouarfate et al. (2021), ATM 
 运行耗时: 0.175973 秒
------------------------------
Bias: [ 5.2073e-04 -1.5333e-04  8.9288e-05] | dt: None
[Case III]: in Kouarfate et al. (2021), ATM 
 运行耗时: 0.239923 秒
------------------------------
Bias: [-0.0005  0.0671  0.0002  0.0002  0.0002] | dt: None
[Case IV]: Near expiration and atm 
 运行耗时: 8.818794 秒
------------------------------
Bias: [0.0006 0.0008 0.0012] | dt: None
[Case V]: Near expiration only 
 运行耗时: 8.593675 秒
------------------------------
Bias: [0.0089 0.0087 0.0086 0.0084 0.0082] | dt: None
[Case VI]: atm only 
 运行耗时: 0.103311 秒
------------------------------
Bias: [0.001 0.001 0.001] | dt: None
[Case VII]: Lewis AL (2000) Option valuation under stochastic volatility: with Mathematica code. Finance 

# Pricing With Exact simulation of stochastic volatility models based on conditional Fourier-cosine method

In [19]:
model = pfex.Sv32McBrignoneJunike2026ConditionalCos
model.use_cos = True
model.n_path = 10e6
run_valuation_for_single_model(model, None, case_names, case_dict)


========== 正在运行模型: Sv32McBrignoneJunike2026ConditionalCos ==========
Bias: -0.04976959391371655 | dt: None
[Case I]: Eq. (4.2) in Baldeaux (2012), ATM 
 运行耗时: 29.460042 秒
------------------------------


c:\Users\27261\Desktop\3_Courses in PHBS\3_09_AppliedStochasticProcess\Project_sv32_EMC\pyfeng\sv32_mc2.py:718: RuntimeWarning: invalid value encountered in sqrt
  N_cos = 128  # COS 级数项，128 足以兼顾高精度与极速
c:\Users\27261\Desktop\3_Courses in PHBS\3_09_AppliedStochasticProcess\Project_sv32_EMC\pyfeng\sv32_mc2.py:789: RuntimeWarning: invalid value encountered in log
  


Bias: [nan nan nan] | dt: None
[Case II]: Set 2 in Kouarfate et al. (2021), ATM 
 运行耗时: 8.316209 秒
------------------------------
Bias: [nan nan nan] | dt: None
[Case III]: in Kouarfate et al. (2021), ATM 
 运行耗时: 5.882246 秒
------------------------------
Bias: [nan nan nan nan nan] | dt: None
[Case IV]: Near expiration and atm 
 运行耗时: 5.829708 秒
------------------------------
[Case V]: Near expiration only 
 运行耗时: 5.259363 秒


KeyboardInterrupt: 

In [ ]:
model = pfex.Sv32McBrignoneJunike2026ConditionalCos
model.use_cos = False
run_valuation_for_single_model(model, None, case_names, case_dict)


========== 正在运行模型: Sv32McBrignoneJunike2026ConditionalCos ==========
Bias: -0.05046968413422509 | dt: None
[Case I]: Eq. (4.2) in Baldeaux (2012), ATM 
 运行耗时: 0.025369 秒
------------------------------
Bias: [nan nan nan] | dt: None
[Case II]: Set 2 in Kouarfate et al. (2021), ATM 
 运行耗时: 0.033640 秒
------------------------------
Bias: [nan nan nan] | dt: None
[Case III]: in Kouarfate et al. (2021), ATM 
 运行耗时: 0.032338 秒
------------------------------
Bias: [nan nan nan nan nan] | dt: None
[Case IV]: Near expiration and atm 
 运行耗时: 0.025736 秒
------------------------------
Bias: [nan nan nan] | dt: None
[Case V]: Near expiration only 
 运行耗时: 0.025943 秒
------------------------------
Bias: [nan nan nan nan nan] | dt: None
[Case VI]: atm only 
 运行耗时: 0.034303 秒
------------------------------
Bias: [nan nan nan] | dt: None
[Case VII]: Lewis AL (2000) Option valuation under stochastic volatility: with Mathematica code. Finance Press 
 运行耗时: 0.032151 秒
------------------------------


# Results Comparison

In [5]:
table1_results = {}
dt_list = [1/50, 1/500, 1/5000]

# 遍历 dt 列表
for dt_val in dt_list:
    run_valuation(pfex.Sv32McTimeStep, 1, "Euler/Milstein scheme", case_names, case_dict, table1_results, dt=dt_val, add_dt_to_index=True)
    run_valuation(pfex.Sv32McTimeStep, 2, "Exact Stepping", case_names, case_dict, table1_results, dt=dt_val, add_dt_to_index=True)
    run_valuation(pfex.Sv32McTimeStep, 3, "Almost Exact Stepping(Poison-Gamma)", case_names, case_dict, table1_results, dt=dt_val, add_dt_to_index=True)
    run_valuation(pfex.Sv32McTimeStep, 4, "Almost Exact Stepping(QE)", case_names, case_dict, table1_results, dt=dt_val, add_dt_to_index=True)

# 转换为 DataFrame 并设置三级行索引
df_table1 = pd.DataFrame.from_dict(table1_results, orient='index')
df_table1.columns = pd.MultiIndex.from_tuples(df_table1.columns)
df_table1.index = pd.MultiIndex.from_tuples(df_table1.index, names=["Case", "dt", "Strike"])

# 如果想让相同 Case 下的顺序是 dt -> Strike，可以使用 sort_index 整理一下层级
df_table1 = df_table1.sort_index(level=["Case", "dt"])

print("\n--- 表格 1 (TimeStep & dt) 预览 ---")
print(df_table1.head())
df_table1.to_excel("table1_timestep_dts.xlsx")


========== 正在运行模型: Sv32McTimeStep - Euler/Milstein scheme ==========
[Case I]: 运行耗时: 0.405230 秒 | dt: 1/50
[Case II]: 运行耗时: 0.267766 秒 | dt: 1/50
[Case III]: 运行耗时: 0.238946 秒 | dt: 1/50
[Case IV]: 运行耗时: 0.089172 秒 | dt: 1/50
[Case V]: 运行耗时: 0.087302 秒 | dt: 1/50
[Case VI]: 运行耗时: 0.383123 秒 | dt: 1/50
[Case VII]: 运行耗时: 0.210370 秒 | dt: 1/50
------------------------------

========== 正在运行模型: Sv32McTimeStep - Exact Stepping ==========
[Case I]: 运行耗时: 0.428566 秒 | dt: 1/50
[Case II]: 运行耗时: 0.245517 秒 | dt: 1/50
[Case III]: 运行耗时: 0.241762 秒 | dt: 1/50
[Case IV]: 运行耗时: 0.083612 秒 | dt: 1/50
[Case V]: 运行耗时: 0.084547 秒 | dt: 1/50
[Case VI]: 运行耗时: 0.430224 秒 | dt: 1/50
[Case VII]: 运行耗时: 0.249706 秒 | dt: 1/50
------------------------------

========== 正在运行模型: Sv32McTimeStep - Almost Exact Stepping(Poison-Gamma) ==========
[Case I]: 运行耗时: 0.620976 秒 | dt: 1/50
[Case II]: 运行耗时: 0.465489 秒 | dt: 1/50
[Case III]: 运行耗时: 0.387025 秒 | dt: 1/50
[Case IV]: 运行耗时: 0.106155 秒 | dt: 1/50
[Case V]: 运行耗时: 0.1

In [ ]:
table2_results = {}
# 跑其它模型 (dt 保持 None，不把 dt 加入索引)
run_valuation(pfex.Sv32McBaldeaux2012Exact, None, "Exact Simulation", case_names, case_dict, table2_results)

model_cos = pfex.Sv32McBrignoneJunike2026ConditionalCos
model_cos.use_cos = True
run_valuation(model_cos, None, "Exact Simulation(Use Fourier-Cosine)", case_names, case_dict, table2_results)

model_cos.use_cos = False
run_valuation(model_cos, None, "Exact Simulation(Use Moment Matching)", case_names, case_dict, table2_results)

model = pfex.Sv32McChoiKwok2023Ig
model.dist = "ig"
run_valuation(model, None, "Almost Exact Simulation(Inverse Gaussian)", case_names, case_dict, table2_results)

model.dist = "ga"
run_valuation(model, None, "Almost Exact Simulation(Gamma)", case_names, case_dict, table2_results)


run_valuation(pf.sv_fft.Sv32Fft, None, "Fast Fourier Transformation", case_names, case_dict, table2_results)
run_valuation(pf.sv_fft.Sv32FourierCos, None, "Fourier-Cosine Method", case_names, case_dict, table2_results)

# 转换为 DataFrame 并设置两级行索引
df_table2 = pd.DataFrame.from_dict(table2_results, orient='index')
df_table2.columns = pd.MultiIndex.from_tuples(df_table2.columns)
df_table2.index = pd.MultiIndex.from_tuples(df_table2.index, names=["Case", "Strike"])

print("\n--- 表格 2 (All Models 对比) 预览 ---")
print(df_table2.head())
df_table2.to_excel("table2_all_models.xlsx")


========== 正在运行模型: Sv32Fft - Fast Fourier Transformation ==========
[Case I]: 运行耗时: 0.349416 秒 | dt: N/A
[Case II]: 运行耗时: 0.183765 秒 | dt: N/A
[Case III]: 运行耗时: 0.269013 秒 | dt: N/A
[Case IV]: 运行耗时: 14.420766 秒 | dt: N/A
[Case V]: 运行耗时: 14.517569 秒 | dt: N/A
[Case VI]: 运行耗时: 0.203815 秒 | dt: N/A
[Case VII]: 运行耗时: 0.297039 秒 | dt: N/A
------------------------------

========== 正在运行模型: Sv32FourierCos - Fourier-Cosine Method ==========
[Case I]: 运行耗时: 0.216868 秒 | dt: N/A
[Case II]: 运行耗时: 0.102573 秒 | dt: N/A
[Case III]: 运行耗时: 0.139744 秒 | dt: N/A
[Case IV]: 运行耗时: 6.928721 秒 | dt: N/A
[Case V]: 运行耗时: 6.865949 秒 | dt: N/A
[Case VI]: 运行耗时: 0.087652 秒 | dt: N/A
[Case VII]: 运行耗时: 0.129311 秒 | dt: N/A
------------------------------

--- 表格 2 (All Models 对比) 预览 ---
                Fast Fourier Transformation                       \
                                    Time(s)          Option Bias   
Case     Strike                                                    
Case I   1.0                

PermissionError: [Errno 13] Permission denied: 'table2_all_models_debug.xlsx'

In [ ]:
format_excel_time_cells("table1_timestep_dts.xlsx")
format_excel_time_cells("table2_all_models.xlsx")

正在处理排版: table1_timestep_dts.xlsx ...
  [成功] 已完成合并并保存！

正在处理排版: table2_all_models_debug.xlsx ...
  [成功] 已完成合并并保存！

